# Dam Break (2D)

A column of water held in the corner of a tank, released at t = 0 and allowed to
run out under gravity. It is the free-surface benchmark of this family: nothing
about it is periodic, nothing is smooth, and the interesting part -- the front
running along the floor, the jet up the far wall, the sheet folding back over
itself -- is a *surface*, not a field.

Three things follow from that, and they are why this notebook differs from the
periodic cases:

- **The column's proportions are the experiment.** It is
  `fluidWidth * W` wide by `fillRatio * L` tall in the bottom-left corner of a
  `W x L` tank; the shipped `1/6` and `2/3` give a 0.667 x 1.333 column in a
  4 x 2 tank -- the canonical Koshizuka & Oka shape, twice as tall as it is
  wide, with six of its own widths of floor to run out along.
- **Free-surface detection is on** (`fillRatio < 1` turns it on automatically),
  which is what the `surfaceIndicators` panel at the end of this notebook is
  reading.
- **The panels are velocity and particle ID, not velocity and density.** A
  cyclic-coloured particle ID is a dye trace: it is how you see the sheet fold
  over itself, which no scalar field shows. `--plotDensity` adds the density
  panel back between them (`dambreakFields(ctx)` picks).

An obstacle can be dropped into the run-out (`obstacleActive=True`,
`obstacleType` one of the presets), which is what
`../sweeps/dambreak_obstacle.yaml` does.

![](outputs/12-dambreak.gif)

## Every knob, and what it does

The parameters cell below is the whole command line of `12-dambreak.py` written
out: `CaseSpec` fields first, then `dambreakCase.params`. This case has the
longest parameter list in the repo -- it came from `datagen`'s ~65-flag
`generator.py` -- so the table below covers the ones this notebook sets and the
groups it leaves alone; `dambreakCase.params` is the full list, and every key in
it is also a `--flag`.

**Discretisation, time stepping and output** (`CaseSpec` fields)

| field | this notebook | what it does |
|---|---|---|
| `nx` | `128` | particles across `L`; the spacing is `dx = L / nx` |
| `dim` | `2` | this case is 2D |
| `L` | `2.0` | tank *height*; `W` below is its width |
| `n_h` | `4.0` | particles per support radius |
| `kernel` | `Wendland4` | SPH kernel |
| `integrationScheme` | `rungeKutta2` | time integrator |
| `scheme` | `deltaSPH` | the solver itself |
| `tLimit` | `4.0` | simulated end time; the loop runs `tLimit / dt` steps |
| `dt` | *set by the case* | left `None`: `initialConditions` picks it together with the sound speed |
| `adaptiveDt`, `cflFactor`, `minDt` | `True`, `0.3`, `1e-8` | CFL limiter around that `dt` |
| `plot`, `show`, `plotInterval` | `True`, `True`, `10` | render a frame every `plotInterval` steps |
| `store`, `storeMode`, `exportInterval` | `False`, `'trajectory'`, `0.002` | HDF5 export; off here, and when on this case writes one growing `trajectory.h5` rather than a file per frame |

**The case's own parameters** (`--flag` on the script, `params=dict(...)` here)

| parameter | this notebook | what it does |
|---|---|---|
| `W` | `4.0` | tank width; with `L = 2` that is the 4 x 2 tank |
| `band` | `5` | particle layers of wall around the interior domain |
| `fillRatio` | `2/3` | column height as a fraction of `L` |
| `fluidWidth` | `1/6` | column width as a fraction of `W` |
| `semiPeriodic`, `fullyPeriodic` | `False`, `False` | replace the walls with periodicity in x, or in both directions |
| `disableGravity` | `False` | the whole driving force; `True` freezes the column in place |
| `gravityMagnitude`, `gravityDirection` | `9.81`, `[0, -1]` | which way, and how hard, it falls |
| `obstacleActive` | `False` | put a body in the run-out |
| `obstacleType` | `'circleMiddle'` | which preset body: see `buildPresetObstacles` |
| `maxExtent`, `offsetX`, `aoa` | `1/16`, `3/4`, `0.0` | its size, its position along the floor, its angle of attack |
| `rho0`-equivalent (`targetDt`) | `0.0005` | the timestep the run *asks* for; the sound speed follows from the acoustic CFL |
| `plotDensity` | `False` | add the density panel between velocity and particle ID |
| `markerSize`, `plotWidth`, `plotHeight` | `4`, `28`, `8` | plot only -- the tank is much wider than it is tall, hence the wide figure |

Three groups are left at their defaults here and not tabulated, because they
belong to the `datagen` variants of this case rather than to the dam break:
`enableFreestream`/`forcingWidth`/`freeStreamVelocity`,
`enableNoise` and its Perlin knobs, and
`enableKolmogorovForcing`/`kolmogorovForcingAmplitude`/`kolmogorovForcingWavenumber`.
Each is off; turning one on is what makes this case the generator for those
datasets.

**Three things this family does differently from the compressible notebooks**
(they will bite if `../compressible/08-Hydrostatic.ipynb` is copied unread):

1. The IC cell has a **fourth call**, `dambreakCase.initialConditions(ctx, system)`.
   That is where the noise/freestream/forcing options are applied and where
   `setupWeaklyCompressibleTimestep` picks the sound speed and `config.dt`
   *together* from `targetDt`. Skip it and `config.dt` stays `None`.
2. **The loop is `range(nSteps)`.** No case in this family has a `timestep`
   hook, so `dt` is fixed for the whole run after step 1 and `while t < tLimit`
   would be the wrong shape.
3. Plotting calls `buildFieldPlotter`/`refreshFieldPlotter` directly rather than
   `dambreakCase.setupPlot`/`updatePlot`, which go through
   `openWindow`/`pumpEvents` and do not live-update inside a Jupyter cell in
   this environment -- `08-Hydrostatic.ipynb` explains that in full. The field
   list comes from `dambreakFields(ctx)` so `plotDensity` still means something
   here.

Precision note: switching between single and double precision is controlled in
the import cell below. Because precision is set when core modules/kernels are
initialized, any precision change requires a kernel restart.

In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.dambreak import dambreakCase, dambreakFields
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.modules import computeDensities
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on `12-dambreak.py`, made
# explicit and editable here -- the table in the intro cell says what each one
# does. `dambreakCase.defaults`/`.params` are the same values the CLI script
# starts from, and the groups this cell does not name (freestream, noise,
# Kolmogorov forcing) stay at theirs.
spec = CaseSpec(caseName=dambreakCase.name, scheme=dambreakCase.scheme,
                params=dict(dambreakCase.params)) \
    .merged(**dambreakCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=128,
    dim=2,
    L=2.0,

    # --- time stepping ---------------------------------------------------
    tLimit=4.0,

    # --- output --------------------------------------------------------------
    caseName='12-dambreak',
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- the tank's own knobs -------------------------------------------------
    params=dict(
        # the tank: W wide by L tall, with `band` layers of wall around it
        W=4.0, band=5,
        # the column: fluidWidth * W by fillRatio * L, in the bottom-left corner
        fillRatio=2.0 / 3.0, fluidWidth=1.0 / 6.0,
        semiPeriodic=False, fullyPeriodic=False,
        # what drives it
        disableGravity=False, gravityMagnitude=9.81, gravityDirection=[0.0, -1.0],
        # an optional body in the run-out
        obstacleActive=False, obstacleType='circleMiddle',
        maxExtent=1.0 / 16.0, offsetX=3.0 / 4.0, aoa=0.0,
        # the fluid
        targetDt=0.0005,
        # plotting: the tank is much wider than it is tall
        plotDensity=False, markerSize=4, plotWidth=28, plotHeight=8,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`dambreakCase.buildSystem`), not re-derived here.
#
# `initialConditions` is the call the compressible notebooks do not have: it
# applies the noise/freestream/forcing options and is where the sound speed and
# `config.dt` are chosen together from `targetDt`, so skipping it leaves
# `config.dt` unset.
ctx = buildContext(dambreakCase, spec)
dambreakCase.configureScheme(ctx)
system = dambreakCase.buildSystem(ctx)
dambreakCase.initialConditions(ctx, system)
runningState = system.initializeNewState()

kinds = system.state.kinds
print(f'dt = {float(ctx.config.dt):.3e}, '
      f'c0 = {ctx.schemeConfig.fluid.fixedSoundSpeed:.3f}, '
      f'{len(runningState.state.positions)} particles '
      f'({int((kinds == 0).sum())} fluid, {int((kinds != 0).sum())} boundary)')
print(f"free-surface detection: {ctx.schemeConfig.surfaceDetectionConfig.active}")
print(f"column: {spec.param('fluidWidth') * spec.param('W'):.3g} wide x "
      f"{spec.param('fillRatio') * spec.L:.3g} tall in a "
      f"{spec.param('W'):.3g} x {spec.L:.3g} tank")

In [ ]:
# What was actually built: the sampled regions, fluid and boundary, against the
# domain the run happens in. This is where "the column is the wrong shape" or
# "the obstacle is in the wrong place" is visible before spending the run.
figure, axis = plt.subplots(1, 1, figsize=(11, 6), squeeze=False)
plotRegions(ctx.schemeConfig.regions, axis[0, 0], plotFluid=True, plotParticles=True)
domain = ctx.config.domain
axis[0, 0].set_aspect('equal')
axis[0, 0].set_xlim(domain.min[0].item(), domain.max[0].item())
axis[0, 0].set_ylim(domain.min[1].item(), domain.max[1].item())
axis[0, 0].set_title(f'{len(ctx.schemeConfig.regions)} regions, '
                     f'{len(runningState.state.positions)} particles')
figure.tight_layout()

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# `dambreakFields(ctx)` rather than a hardcoded list, so `plotDensity` above
# still means something; direct buildFieldPlotter rather than
# dambreakCase.setupPlot -- see the intro cell for why.
fields = dambreakFields(ctx)
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, fields,
                                figsize=(spec.param('plotWidth'), spec.param('plotHeight')))

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = dambreakCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData,
                              extraFields=dambreakCase.extraFields)

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = [dict(dambreakCase.diagnostics(ctx, runningState), step=-1, t=0.0)]
# The front position: the x of the furthest fluid particle, which is the one
# number this benchmark is usually reported as. Injected at the hook point
# because nothing in the case records it.
frontPosition = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    particles = runningState.state
    fluid = particles.kinds == 0
    frontPosition.append(float(particles.positions[fluid, 0].max()))
    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = dambreakCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, fields, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                   schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                   extraFields=dambreakCase.extraFields)

In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## The free surface, after the fact

Two things the live panels cannot show, both read off the *final* state:

- **The density, recomputed.** `computeDensities` is the plain SPH sum
  $\rho_i = \sum_j m_j W_{ij}$, evaluated fresh rather than carried along by the
  scheme. Near a free surface it necessarily reads low -- a particle at the
  surface is missing half its neighbourhood -- and that deficit *is* the surface,
  which is why this cell is worth keeping even though the run already has a
  density panel available.
- **What the surface detector thinks.** `surfaceIndicators` is the scheme's own
  answer to the same question, and this case has it because `fillRatio < 1`
  turns free-surface detection on. Comparing the two panels is the check: the
  low-density rim and the flagged particles should be the same rim.

The third panel is the standing weakly compressible health check -- the density
bounds against the +-1% band -- plus the front position, which is what a dam
break is usually reported as.

In [ ]:
densities = computeDensities(runningState.state, ctx.config, ctx.schemeConfig, None)
positions = runningState.state.positions.detach().cpu().numpy()
fluid = (runningState.state.kinds == 0).detach().cpu().numpy()
indicators = getattr(runningState.state, 'surfaceIndicators', None)

figure, axis = plt.subplots(2, 1, figsize=(14, 7))
scatter = axis[0].scatter(positions[fluid, 0], positions[fluid, 1],
                          c=densities.detach().cpu().numpy()[fluid], s=1, cmap='viridis')
figure.colorbar(scatter, ax=axis[0])
axis[0].set_title(r'recomputed $\sum_j m_j W_{ij}$ -- the surface reads low')

if indicators is not None:
    scatter = axis[1].scatter(positions[fluid, 0], positions[fluid, 1],
                              c=indicators.detach().cpu().numpy()[fluid], s=1, cmap='viridis')
    figure.colorbar(scatter, ax=axis[1])
    axis[1].set_title('surfaceIndicators -- what the detector flagged')
else:
    axis[1].set_title('no surfaceIndicators on this state (free-surface detection off)')

for ax in axis:
    ax.set_aspect('equal')
    ax.set_xlim(domain.min[0].item(), domain.max[0].item())
    ax.set_ylim(domain.min[1].item(), domain.max[1].item())
figure.tight_layout()

In [ ]:
figure, axis = plt.subplots(1, 2, figsize=(11, 4))
t = np.array([row['t'] for row in trajectory])

axis[0].plot(t[1:], frontPosition)
axis[0].axhline(domain.max[0].item(), color='black', ls=':', lw=0.8, label='far wall')
axis[0].set_xlabel('t'); axis[0].set_ylabel('front position (max fluid x)'); axis[0].legend()

axis[1].plot(t, [row['maxDensity'] for row in trajectory], label='max')
axis[1].plot(t, [row['minDensity'] for row in trajectory], label='min')
axis[1].axhspan(0.99, 1.01, color='green', alpha=0.1, label=r'$\pm 1\%$')
axis[1].set_xlabel('t'); axis[1].set_ylabel(r'$\rho$'); axis[1].legend()
figure.tight_layout()